In [0]:
%pip install -qqqq -U mlflow[genai,databricks] databricks-sdk
dbutils.library.restartPython()

## [Step 1: Configure](https://mlflow.org/cookbook/genie-tracing-pipeline/#step-1-configure)

In [0]:
dbutils.widgets.text("genie_agent_id", "", "Genie Agent ID")
dbutils.widgets.text("experiment_name", "", "Experiment Name")
AGENT_ID = dbutils.widgets.get("genie_agent_id")
EXPERIMENT_NAME = dbutils.widgets.get("experiment_name")

In [0]:
import mlflow
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

mlflow.set_experiment(EXPERIMENT_NAME)

In [0]:
import os

default_warehouse = next(
    (
        wh
        for wh in w.warehouses.list()
        if "Serverless Starter Warehouse" in wh.name and wh.enable_serverless_compute
    ),
    None,
)
default_warehouse_id = default_warehouse.id if default_warehouse else None
print(f"{default_warehouse_id=}")

# Specify the ID of a SQL warehouse you have access to.
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = default_warehouse_id

## [Step 2: Pull Conversations and Log as Traces](https://mlflow.org/cookbook/genie-tracing-pipeline/#step-2-pull-conversations-and-log-as-traces)

In [0]:
# 1. Collect Genie message IDs that have already been traced so we
#    can skip them and safely re-run this pipeline as new
#    conversations come in.
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
existing_traces = mlflow.search_traces(
    locations=[experiment.experiment_id], return_type="list"
)
already_traced = {
    t.info.tags.get("message_id")
    for t in existing_traces
    if t.info.tags.get("message_id")
}

# 2. Pull every conversation from the Genie space.
conversations = w.genie.list_conversations(space_id=AGENT_ID, include_all=True)

# 3. Loop through Genie messages, skip duplicates, and log each
#    new message as an MLflow trace with the question, SQL, and
#    response.
traced = 0
for convo in conversations.conversations or []:
    messages = w.genie.list_conversation_messages(
        space_id=AGENT_ID, conversation_id=convo.conversation_id
    )
    for msg in messages.messages or []:
        if not msg.content:
            continue
        if msg.message_id in already_traced:
            continue

        # 3a. Extract the SQL query and text response from the
        #     Genie message attachments.
        attachments = msg.attachments or []
        sql_att = next((a for a in attachments if a.query), None)
        text_att = next((a for a in attachments if a.text), None)

        # 3b. Log the question, SQL, and response as an MLflow
        #     trace for inspection and evaluation.
        with mlflow.start_span(name="genie_interaction") as span:
            span.set_inputs({"question": msg.content})
            span.set_outputs(
                {
                    "response": (text_att.text.content if text_att else None),
                    "generated_sql": (sql_att.query.query if sql_att else None),
                    "error": str(msg.error) if msg.error else None,
                }
            )
            # 3c. Tag the trace with the Genie message ID so
            #     future runs know this message has already been
            #     traced.
            mlflow.update_current_trace(tags={"message_id": msg.message_id})

        traced += 1

print(f"Logged {traced} new traces to experiment: {EXPERIMENT_NAME}")